# putEMG — Preprocessing & Feature Extraction Driver

Run this notebook whenever you add new subjects or want to regenerate data.

| Stage | Description | Output |
|-------|-------------|--------|
| **1** | Signal preprocessing | `UG_per_subject/` — one `.mat` per subject |
| **2A** | Features — flat per window | `feature_gestures/flat_window/` — `(N_windows, 192)` |
| **2B** | Features — flat per rep | `feature_gestures/flat_rep/` — `(N_reps, 4992)` |
| **2C** | Features — sequence per rep | `feature_gestures/sequence/` — `(N_reps, 26, 192)` |

Stages 2A–2C are independent — run whichever formats you need.
All feature files are saved as `.npz` with keys `X` and `y`.

In [2]:
import scipy.io

---
## Stage 1 — Signal Preprocessing

Processes every raw per-subject `.mat` file:
**bandpass 20–500 Hz** → **resample to 1500 samples** → **per-channel z-score normalization**.

Input : `NUG_per_subject/` — raw MATLAB pipeline output  
Output: `UG_per_subject/` — one processed `.mat` per subject, key `combinedCell (N_reps, 7)`

In [3]:
from preprocessing import batch_process_subjects

RAW_DIR  = '/Volumes/KRIS/data/NUG_per_subject'
PROC_DIR = '/Volumes/KRIS/data/UG_per_subject'

batch_process_subjects(
    input_dir      = RAW_DIR,
    output_dir     = PROC_DIR,
    apply_bandpass = True,   # 20–500 Hz zero-phase Butterworth
    apply_zscore   = True,   # per-channel z-score
    target_length  = 1500,   # samples after resampling
)

Found 45 subject file(s)
Bandpass: ON | Z-score: ON | Target length: 1500

[1/45] ._emg_gestures_13_NU_mat.mat
Loading: ._emg_gestures_13_NU_mat.mat
  ERROR: Mat 4 mopt wrong format, byteswapping problem?

[2/45] emg_gestures_03_NU_mat.mat
Loading: emg_gestures_03_NU_mat.mat
  Shape: 40 repetitions × 7 gestures

=== Gesture Length Statistics ===
  G1: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G2: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G3: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  G6: min=  5124  max= 15373  mean= 10248.3  median= 10248  (n=40)
  G7: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  G8: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G9: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  Processed 280 repetitions → shape (1500 × 24) each
  Saved → /Volumes/KRIS/data/UG_per_subject/emg_gestures_03_U.mat

[3/45] emg_gestures_04_NU_mat.mat
Loading: emg_gestures_04_NU_m

KeyboardInterrupt: 

---
## Stage 2 — Feature Extraction

Extracts **8 features × 24 channels = 192 features** per sliding window.

```
window_size  = 250 samples  (~49 ms at 5120 Hz)
window_shift = 50  samples  (~10 ms stride)
windows/rep  = 26
```

Features: MAV · RMS · WL · ZC · SSC · VAR · MNF · MDF  
Run 2A, 2B, and 2C independently — they do not depend on each other.

### 2A — Flat Per Window (`flat_window`)

Each sliding window is one independent sample.  
Output shape: `X (N_windows, 192)`, `y (N_windows,)` — ~7 280 samples per subject.  
Use for: **SVM, XGBoost, MLP** (real-time capable).

In [4]:
from feature_extraction import batch_extract_features

PROC_DIR = '/Volumes/KRIS/data/UG_per_subject'
OUT_DIR  = '/Volumes/KRIS/data/feature_gestures/flat_window'

batch_extract_features(
    input_dir     = PROC_DIR,
    output_dir    = OUT_DIR,
    mode          = 'flat_window',
    window_size   = 250,
    window_shift  = 50,
    sampling_rate = 5120.0,
)
# Output: features_subject_<ID>_flat_window.npz  (keys: X, y)

Found 44 .mat file(s) in:
  /Volumes/KRIS/data/UG_per_subject
Mode  : flat_window

─── emg_gestures_03_U.mat  →  features_subject_03_flat_window.npz
Loaded  : emg_gestures_03_U.mat
Shape   : 40 repetitions × 7 gestures
Mode    : flat_window
Window  : size=250, shift=50, fs=5120.0 Hz
Features: 8 per channel × 24 channels = 192 total

  Gesture G1 (class 0) — accumulated 40 reps so far
  Gesture G2 (class 1) — accumulated 80 reps so far
  Gesture G3 (class 2) — accumulated 120 reps so far
  Gesture G6 (class 3) — accumulated 160 reps so far
  Gesture G7 (class 4) — accumulated 200 reps so far
  Gesture G8 (class 5) — accumulated 240 reps so far
  Gesture G9 (class 6) — accumulated 280 reps so far

Windows per rep : 26
Feature matrix  : X=(7280, 192), y=(7280,)
Class distribution: {'G1': 1040, 'G2': 1040, 'G3': 1040, 'G6': 1040, 'G7': 1040, 'G8': 1040, 'G9': 1040}

Saved   : /Volumes/KRIS/data/feature_gestures/flat_window/features_subject_03_flat_window.npz

─── emg_gestures_04_U.mat  →  

### 2B — Flat Per Rep (`flat_rep`)

All windows from one rep are concatenated into a single flat vector.  
Output shape: `X (N_reps, 4992)`, `y (N_reps,)` — ~280 samples per subject.  
Use for: **SVM, XGBoost, MLP** (offline only — needs full gesture first).

> Requires every rep to produce the same number of windows.  
> With `window_size=250, shift=50, length=1500` this gives exactly 26 windows/rep.

In [6]:
from feature_extraction import batch_extract_features

PROC_DIR = '/Volumes/KRIS/data/UG_per_subject'
OUT_DIR  = '/Volumes/KRIS/data/feature_gestures/flat_rep'

batch_extract_features(
    input_dir     = PROC_DIR,
    output_dir    = OUT_DIR,
    mode          = 'flat_rep',
    window_size   = 250,
    window_shift  = 50,
    sampling_rate = 5120.0,
)
# Output: features_subject_<ID>_flat_rep.npz  (keys: X, y)

Found 44 .mat file(s) in:
  /Volumes/KRIS/data/UG_per_subject
Mode  : flat_rep

─── emg_gestures_03_U.mat  →  features_subject_03_flat_rep.npz
Loaded  : emg_gestures_03_U.mat
Shape   : 40 repetitions × 7 gestures
Mode    : flat_rep
Window  : size=250, shift=50, fs=5120.0 Hz
Features: 8 per channel × 24 channels = 192 total

  Gesture G1 (class 0) — accumulated 40 reps so far
  Gesture G2 (class 1) — accumulated 80 reps so far
  Gesture G3 (class 2) — accumulated 120 reps so far
  Gesture G6 (class 3) — accumulated 160 reps so far
  Gesture G7 (class 4) — accumulated 200 reps so far
  Gesture G8 (class 5) — accumulated 240 reps so far
  Gesture G9 (class 6) — accumulated 280 reps so far

Windows per rep : 26
Feature matrix  : X=(280, 4992), y=(280,)
Class distribution: {'G1': 40, 'G2': 40, 'G3': 40, 'G6': 40, 'G7': 40, 'G8': 40, 'G9': 40}

Saved   : /Volumes/KRIS/data/feature_gestures/flat_rep/features_subject_03_flat_rep.npz

─── emg_gestures_04_U.mat  →  features_subject_04_flat_rep.n

### 2C — Sequence Per Rep (`sequence`)

Windows are stacked in temporal order — one sequence matrix per rep.  
Output shape: `X (N_reps, 26, 192)`, `y (N_reps,)` — ~280 samples per subject.  
Use for: **LSTM, GRU, Transformer** (offline only).

> Same window-count requirement as 2B.

In [5]:
from feature_extraction import batch_extract_features

PROC_DIR = '/Volumes/KRIS/data/UG_per_subject'
OUT_DIR  = '/Volumes/KRIS/data/feature_gestures/sequence'

batch_extract_features(
    input_dir     = PROC_DIR,
    output_dir    = OUT_DIR,
    mode          = 'sequence',
    window_size   = 250,
    window_shift  = 50,
    sampling_rate = 5120.0,
)
# Output: features_subject_<ID>_sequence.npz  (keys: X, y)

Found 44 .mat file(s) in:
  /Volumes/KRIS/data/UG_per_subject
Mode  : sequence

─── emg_gestures_03_U.mat  →  features_subject_03_sequence.npz
Loaded  : emg_gestures_03_U.mat
Shape   : 40 repetitions × 7 gestures
Mode    : sequence
Window  : size=250, shift=50, fs=5120.0 Hz
Features: 8 per channel × 24 channels = 192 total

  Gesture G1 (class 0) — accumulated 40 reps so far
  Gesture G2 (class 1) — accumulated 80 reps so far
  Gesture G3 (class 2) — accumulated 120 reps so far
  Gesture G6 (class 3) — accumulated 160 reps so far
  Gesture G7 (class 4) — accumulated 200 reps so far
  Gesture G8 (class 5) — accumulated 240 reps so far
  Gesture G9 (class 6) — accumulated 280 reps so far

Windows per rep : 26
Feature matrix  : X=(280, 26, 192), y=(280,)
Class distribution: {'G1': 40, 'G2': 40, 'G3': 40, 'G6': 40, 'G7': 40, 'G8': 40, 'G9': 40}

Saved   : /Volumes/KRIS/data/feature_gestures/sequence/features_subject_03_sequence.npz

─── emg_gestures_04_U.mat  →  features_subject_04_sequenc

---
## Verify Outputs

Quick sanity check — print shapes for one file from each format.

In [ ]:
import glob, os
import numpy as np

FEAT_BASE = '/Volumes/KRIS/data/feature_gestures'

for mode in ('flat_window', 'flat_rep', 'sequence'):
    files = sorted(glob.glob(os.path.join(FEAT_BASE, mode, f'*_{mode}.npz')))
    if not files:
        print(f'{mode:15s}  — no files found')
        continue
    data = np.load(files[0])
    print(f'{mode:15s}  {len(files):>3} subjects  '
          f'X={data["X"].shape}  y={data["y"].shape}  '
          f'(from {os.path.basename(files[0])})')